In [ ]:
import json
from datetime import datetime
import pycountry
from matplotlib import pyplot as plt
import pycountry_convert as pc

from emu_renewal.inputs import DATA_PATH, get_dec_pooled_totals, get_continent_data, get_continent_vars, get_country_vars, extract_specific_var
from emu_renewal.run import find_run_end_time, get_country_vacc_data

In [ ]:
countries = json.load(open(DATA_PATH / f"config/included.json", "r"))
date_threshold = datetime(2021, 4, 1)
vacc_thresh = 0.05
delta_countries = []
for c in countries:
    vacc_data = get_country_vacc_data(c)
    end_time = find_run_end_time(vacc_data, vacc_thresh, c)
    if end_time > date_threshold:
        delta_countries.append(c)

In [ ]:
iso3 = "FRA"
iso2 = pycountry.countries.lookup(iso3).alpha_2
continent = pc.country_alpha2_to_continent_code(iso2)
var_data = get_country_vars(iso3)
alpha_data = extract_specific_var(var_data, "alpha")

In [ ]:
def get_alpha_target(var_data, continent, end_time):
    alpha_period_start = datetime(2020, 1, 1)
    alpha_period_end = datetime(2021, 6, 1)
    if continent != "OC":
        alpha_data = extract_specific_var(var_data, "alpha")
        if alpha_data is None:
            cont_data = get_continent_data(continent, "alpha")
            alpha_data = get_continent_vars(cont_data, "alpha")
        filt_data = alpha_data[(alpha_period_start < alpha_data.index) & (alpha_data.index < alpha_period_end)]
        pooled_data = get_dec_pooled_totals(filt_data, "alpha")
    return pooled_data["alpha_prop"]

In [ ]:
get_alpha_target(var_data, continent, end_time).plot()

In [ ]:
from emu_renewal.utils import get_row_proportions

In [ ]:
early_data = var_data[var_data.index < datetime(2021, 4, 1)]
early_data = early_data[[c for c in early_data.columns if early_data[c].sum() > 0.0]]
get_row_proportions(early_data).plot.area()

In [ ]:

for c in delta_countries[:1]:
    country = pycountry.countries.lookup(c).name
    print(country)
    var_data = get_country_vars(c)
    delta_data = extract_specific_var(var_data, "delta")
    if delta_data is None:
        iso2 = pycountry.countries.lookup(c).alpha_2
        continent = pc.country_alpha2_to_continent_code(iso2)
        cont_data = get_continent_data(continent, "delta")
        delta_data = get_continent_vars(cont_data, "delta")
    filt_data = delta_data[(start < delta_data.index) & (delta_data.index < end)]
    pooled_data = get_dec_pooled_totals(filt_data, "delta")
    plt.figure()
    filt_data["delta_prop"].plot()
    pooled_data["delta_prop"].plot()
    plt.title(country)